In [5]:
from pathlib import Path
BASE = Path("/home/lka/HYBRID_SARIMA")
print(BASE.exists(), BASE)

#Import the file to read it
import pandas as pd
disocupati_trimestral = pd.read_csv(BASE / 'data.csv', 
                       index_col='Period', 
                       parse_dates=True)
print(disocupati_trimestral)

True /home/lka/HYBRID_SARIMA
             15-34 M    15-34 F   35-49 M   35-49 F   50-74 M   50-74 F  \
Period                                                                    
2004-Q1    11.720628  17.198032  4.283283  8.776062  4.004764  5.376203   
2004-Q2    11.410664  16.187372  3.654493  7.196378  3.636536  4.515018   
2004-Q3    10.041510  15.682014  3.520125  6.945425  3.394804  3.976243   
2004-Q4    11.918557  17.227696  4.020448  7.582207  3.604743  3.583677   
2005-Q1    12.334781  16.852636  4.316241  7.577546  3.440298  4.204936   
...              ...        ...       ...       ...       ...       ...   
2025-Q2    11.341464  13.635416  5.106017  7.025122  3.776334  3.889283   
2025-Q3     9.628072  11.355385  4.656108  6.239399  3.108541  3.482718   
2025-Q4    10.903735  10.928362  3.989899  5.704370  3.250742  3.129547   
2026-Q1     9.534584  10.900056  4.316380  5.611548  2.984988  3.392216   
2026-Q2    10.244865  11.855438  5.044973  6.388627  3.268385  3.460483

/tmp/ipykernel_26336/3099411105.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  disocupati_trimestral = pd.read_csv(BASE / 'data.csv',


In [6]:
#Dividir el dataset en training, validation y test
series = [
    '15-34 M', '15-34 F',
    '35-49 M', '35-49 F',
    '50-74 M', '50-74 F',
    'Global_Rate'
]

# Ordenar cronológicamente
df = disocupati_trimestral.sort_index()

# División 70% Train, 15% Validation, 15% Test
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df[series].iloc[:train_end].copy()
val = df[series].iloc[train_end:val_end].copy()
test = df[series].iloc[val_end:].copy()

# Mostrar resultados para cada serie
#for serie in series:
#    print(f"\nSerie: {serie}")
#    print(f"Train: {len(train[serie])} observaciones")
#    print(f"Validation: {len(val[serie])} observaciones")
#    print(f"Test: {len(test[serie])} observaciones")

print(train)
print(val)
print(test)


             15-34 M    15-34 F   35-49 M    35-49 F   50-74 M   50-74 F  \
Period                                                                     
2004-Q1    11.720628  17.198032  4.283283   8.776062  4.004764  5.376203   
2004-Q2    11.410664  16.187372  3.654493   7.196378  3.636536  4.515018   
2004-Q3    10.041510  15.682014  3.520125   6.945425  3.394804  3.976243   
2004-Q4    11.918557  17.227696  4.020448   7.582207  3.604743  3.583677   
2005-Q1    12.334781  16.852636  4.316241   7.577546  3.440298  4.204936   
...              ...        ...       ...        ...       ...       ...   
2018-Q2    17.908050  21.321705  7.759217  10.435680  6.683032  6.283515   
2018-Q3    16.802144  19.343542  6.191831   9.120602  5.538136  5.299045   
2018-Q4    19.322895  22.132355  7.396393  10.673055  6.327076  6.189145   
2019-Q1    19.163606  21.713279  8.111591  11.250865  6.697788  6.034775   
2019-Q2    17.128382  19.347371  7.161324   9.818228  5.738406  5.356958   

           

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import itertools
import warnings
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

pq=list(itertools.product(range(3),repeat=2))
d=range(3)
pdq=[(p,d_val,q) for (p,q) in pq for d_val in d]
seasonal_pdq=[(P,D,Q,4) for P,D,Q in pdq]
sarima_models,results={},[]
for col in train.columns:
    y_train=train[col].dropna().reset_index(drop=True)
    y_val=val[col].dropna().reset_index(drop=True)
    candidatos=[]
    for order in pdq:
        for seasonal_order in seasonal_pdq:
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    model=SARIMAX(y_train,order=order,seasonal_order=seasonal_order,enforce_stationarity=True,enforce_invertibility=True)
                    fitted_model=model.fit(disp=False)
                if fitted_model.mle_retvals.get("converged",False) and np.isfinite(fitted_model.bic):
                    candidatos.append((fitted_model.bic,fitted_model))
            except Exception:
                continue
    sarima_models[col]=[]
    for bic,model in sorted(candidatos,key=lambda x:x[0]):
        if len(sarima_models[col])==5:
            break
        try:
            pred=np.asarray(model.forecast(steps=len(y_val)))
            fitted=np.asarray(model.fittedvalues)
            if not np.all(np.isfinite(pred)) or not np.all(np.isfinite(fitted)):
                continue
            real=y_val.to_numpy()
            values=y_train.to_numpy()
            mae=mean_absolute_error(real,pred)
            mse=mean_squared_error(real,pred)
            mask=real!=0
            mape=np.mean(np.abs((real[mask]-pred[mask])/real[mask]))*100 if mask.any() else np.nan
            scale=np.mean(np.abs(values[4:]-values[:-4])) if len(values)>4 else np.nan
            sarima_models[col].append(model)
            lb_test=acorr_ljungbox(model.resid,return_df=True)
            lb_pvalue=lb_test["lb_pvalue"].values[-1] if not lb_test.empty else np.nan
            jb_test=jarque_bera(model.resid)
            jb_pvalue=jb_test[1] if jb_test is not None else np.nan
            hetero_test=het_arch(model.resid)
            hetero_pvalue=hetero_test[1] if hetero_test is not None else np.nan
            results.append({
                "Serie":col,"Rank":len(sarima_models[col]),
                "Order":model.model.order,"Seasonal_Order":model.model.seasonal_order,
                "AIC":model.aic,"BIC":bic,"MAE":mae,
                "MSE":mse,"RMSE":np.sqrt(mse),"MAPE":mape,
                "MASE":mae/scale if np.isfinite(scale) and scale>0 else np.nan,
                "RMSEE":np.sqrt(np.mean((values-fitted)**2)),
                "R2":r2_score(real,pred) if len(real)>1 else np.nan,
                "Ljung-Box (p-value)": lb_pvalue,
                "Jarque-Bera (p-value)": jb_pvalue,
                "Heteroscedasticity (p-value)": hetero_pvalue
            })
        except (ValueError,FloatingPointError,OverflowError):
            continue
    print(f"{col} (ok)")
    
top5_sarima=pd.DataFrame(results).round(4)
(BASE/"results").mkdir(parents=True,exist_ok=True)
top5_sarima.to_csv(BASE/"results"/"top5_sarima.csv",index=False)
display(top5_sarima)

/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarn

15-34 M (ok)


/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarn

15-34 F (ok)


/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarn

35-49 M (ok)


... (4 outputs left out)

/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarn

50-74 F (ok)
Global_Rate (ok)


/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarning: acorr_lm currently returns a plain tuple whose length depends on the store argument. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an LMTestResult NamedTuple. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  return acorr_lm(
/home/lka/.venv/lib/python3.12/site-packages/statsmodels/stats/diagnostic.py:997: FutureWarn

,Serie,Rank,Order,Seasonal_Order,AIC,BIC,MAE,MSE,RMSE,MAPE,MASE,RMSEE,R2,Ljung-Box (p-value),Jarque-Bera (p-value),Heteroscedasticity (p-value)
0,15-34 M,1,"(0, 1, 0)","(0, 1, 1, 4)",160.1023,164.1884,1.8727,4.5042,2.1223,12.7907,1.1556,1.8953,0.1351,0.6716,0.0000,0.4981
1,15-34 M,2,"(0, 1, 0)","(1, 1, 1, 4)",160.9989,167.1280,2.7996,11.6219,3.4091,20.0194,1.7275,1.8894,-1.2316,0.6070,0.0000,0.6399
2,15-34 M,3,"(0, 1, 0)","(0, 1, 2, 4)",161.4346,167.5637,2.6092,9.9651,3.1568,18.5897,1.6100,1.8907,-0.9135,0.6401,0.0000,0.5851
3,15-34 M,4,"(1, 1, 0)","(0, 1, 1, 4)",161.9804,168.1095,1.9433,4.9507,2.2250,13.3797,1.1991,1.8943,0.0494,0.6598,0.0000,0.4091
4,15-34 M,5,"(0, 1, 1)","(0, 1, 1, 4)",162.0036,168.1327,1.9309,4.8683,2.2064,13.2767,1.1915,1.8945,0.0652,0.6622,0.0000,0.4259
5,15-34 F,1,"(0, 1, 0)","(0, 1, 1, 4)",168.4573,172.5434,1.5209,3.4749,1.8641,8.0264,0.9468,2.6584,0.0227,0.4698,0.0000,0.9471
6,15-34 F,2,"(0, 1, 0)","(1, 1, 1, 4)",169.6670,175.7962,1.9044,4.7914,2.1889,10.5695,1.1856,2.6555,-0.3476,0.4316,0.0000,0.9053
7,15-34 F,3,"(0, 1, 0)","(0, 1, 2, 4)",169.7404,175.8695,1.8356,4.3981,2.0972,10.1123,1.1427,2.6557,-0.2370,0.4381,0.0000,0.9183
8,15-34 F,4,"(0, 1, 1)","(0, 1, 1, 4)",170.2446,176.3738,1.4982,3.5139,1.8745,7.8642,0.9327,2.6581,0.0117,0.4763,0.0000,0.9516
9,15-34 F,5,"(1, 1, 0)","(0, 1, 1, 4)",170.2852,176.4144,1.5010,3.5064,1.8725,7.8849,0.9345,2.6582,0.0138,0.4750,0.0000,0.9500
